# ReplaceMe on Llama-3-8B: prune → heal → restore → evaluate

Everything is driven by `configs/llama3_8b_nf4.yaml`: paths, the fixed protocol (nf4, LoRA r8 on the MLP,
one epoch, HF generate with 256 new tokens) and the list of experiments (which blocks go). Stages skip work
that is already on disk, so after a kernel restart just run cell 1 again and continue.

Memory on the 2080 Ti is tight. The rule: **one heavy stage per kernel** (profile, prune, train, evaluate),
restart in between. Cells 10–12 need no GPU.

| Cell | What | Kernel |
|---|---|---|
| 1 | config, ReplaceMe import (patches applied in place) | after every restart |
| 2–4 | baseline eval, SFT control train, SFT eval | restart after each |
| 5 | profiles for every window length the experiments need | restart after |
| 6 | blocks of every experiment → `plan.json` | — |
| 7 | prune every experiment (lstsq_joint, both cards) | restart after |
| 8 | heal every experiment (LoRA on the pruned model) | restart after each model if memory does not come back |
| 9 | restore + evaluate (with T; without T where asked) | restart after |
| 10 | summary + McNemar | — |
| 11 | signal analysis: rank tables, greedy choice per signal | — |
| 12 | diagnostic: per-token grad_contrib (GPU, ~5 min) | fresh kernel |

In [ ]:
# 1. Config and the ReplaceMe modules (run after every kernel restart)
import sys
from pathlib import Path
ROOT = Path.cwd().parent if (Path.cwd().parent / "rmexp").is_dir() else Path.cwd()
sys.path.insert(0, str(ROOT))                       # or: pip install -e <repo> once, then drop this line

import gc, json, torch, pandas as pd
from rmexp import load_config, Experiment, resolve_blocks, vram, free_disk_gb
from rmexp.replaceme_repo import load_replaceme
from rmexp.pipeline import eval_baseline, train_sft, eval_sft, needed_lengths
from rmexp.profile import profile, profile_csv
from rmexp import stats, analysis

CONFIG = ROOT / "configs" / "llama3_8b_nf4.yaml"
cfg, specs = load_config(CONFIG)
rm = load_replaceme(cfg)                            # ds = distance_scored, lj = lstsq_joint, patched and bound
print("runs:", cfg.runs_dir, "| free disk GB:", free_disk_gb(cfg.runs_dir), "|", torch.cuda.get_device_name(0))
print("experiments:", [s.name for s in specs]); vram()

In [ ]:
# 2. Baseline: the model as is
eval_baseline(cfg)

In [ ]:
# 3. SFT control: LoRA on the full model with the heal protocol (~40 min)
train_sft(cfg)

In [ ]:
# 4. SFT eval: 4-bit base + adapter, no merge (~40 min)
eval_sft(cfg)

In [ ]:
# 5. Profiles: dist_act, dist_grad (+ grad_contrib, grad_contrib_fro, grad_coher) for every window length
#    the experiments need. ~1-2 min per length on one card; CSVs under runs/profiles are reused.
lengths = needed_lengths(specs)
print("window lengths:", lengths)
profiles = {k: profile(cfg, rm, k) for k in lengths}
for k, p in profiles.items():
    print(f"\nskip {k}: top-5 by the combined rank score")
    from rmexp.profile import ranked
    print(ranked(p).head(5)[["block_start", "block_end", "dist_act", "dist_grad", "rank_act", "rank_grad", "score_ras"]].to_string(index=False))

In [ ]:
# 6. Blocks of every experiment (explicit, from plan.json, or chosen from the profiles) -> plan.json
profiles = {k: pd.read_csv(profile_csv(cfg, k)) for k in needed_lengths(specs) if profile_csv(cfg, k).exists()}
exps = [Experiment(cfg, s, resolve_blocks(cfg, s, profiles)) for s in specs]
print(pd.DataFrame([{"experiment": e.spec.name, "blocks (1-based)": e.blocks, "deleted (0-based)": e.deleted,
                     "T in layers": e.carriers} for e in exps]).to_string(index=False))

In [ ]:
# 7. Prune: one T per block, all T fitted on the original model in one pass. Both cards (decoder on GPU0,
#    lm_head on GPU1). Norm ratio ||T^T W|| / ||W|| of every carrier is printed; above 3.1 the block did not recover.
for e in exps:
    e.prune(rm)
    if e.spec.eval_pruned:
        print("  (T-only eval is done in cell 9 to keep this kernel light)")

In [ ]:
# 8. Heal: LoRA r8 on the pruned model (~35-40 min each). If memory does not come back between models,
#    restart the kernel, run cell 1 and 6, and run this cell again: finished adapters are skipped.
#    A heal that OOMs on one card: set train_lm_head_on_second_gpu: true in the YAML (four carriers needed it).
for e in exps:
    gc.collect(); torch.cuda.empty_cache()
    assert vram(f"before heal {e.spec.name}") < 0.5, "restart the kernel, run 1 and 6, then this cell"
    e.heal()

In [ ]:
# 9. Restore the deleted layers (CPU, two models in RAM) and evaluate (~35-45 min per model).
#    T-only / healed evals for experiments with eval_pruned: true run here as well. Restored checkpoints
#    (16 GB each) are deleted after evaluation; they can be rebuilt from pruned + adapter at any time.
DELETE_RESTORED = True
for e in exps:
    if e.spec.eval_pruned:
        e.eval_pruned()
        e.eval_healed()
    e.eval_restored(remove_T=False, delete_after=DELETE_RESTORED)
    if e.spec.restore_without_T:
        e.eval_restored(remove_T=True, delete_after=DELETE_RESTORED)

In [ ]:
# 10. Summary: every restored model against fine-tuned, and against the first experiment (McNemar on predictions)
from rmexp.evaluate import result_path
runs = {"baseline": (result_path(cfg.results_dir, "baseline", cfg.precision), "—"),
        "fine-tuned": (result_path(cfg.results_dir, "sft", cfg.precision), "—")}
for e in exps:
    files = e.result_files()
    label = " ".join(f"({s},{e_})" for s, e_ in e.blocks)
    if files["replaceme"].exists(): runs[f"restored | {e.spec.name}"] = (files["replaceme"], label)
    if files["replaceme_noT"].exists(): runs[f"restored no T | {e.spec.name}"] = (files["replaceme_noT"], label)
    if files["pruned_healed"].exists(): runs[f"pruned + heal | {e.spec.name}"] = (files["pruned_healed"], label)
    if files["pruned_T_only"].exists(): runs[f"T only | {e.spec.name}"] = (files["pruned_T_only"], label)
pd.set_option("display.width", 220)
print(stats.summary(runs, reference="fine-tuned").to_string(index=False))
first = f"restored | {exps[0].spec.name}"
print("\nagainst", first)
print(stats.pairs({k: v for k, v in runs.items() if k.startswith("restored")}, against=first).to_string(index=False))

In [ ]:
# 11. Signals: Spearman between them, top-3 per score, rank tables with the used blocks marked, greedy choice per signal
profiles = {k: pd.read_csv(profile_csv(cfg, k)) for k in needed_lengths(specs) + [8] if profile_csv(cfg, k).exists()}
if 8 in profiles:
    print("Spearman between signals, 8-layer windows:"); print(analysis.signal_correlations(profiles[8]).to_string())
    print("top-3:", analysis.top_k(profiles[8]))
em = {}
for e in exps:
    f = e.result_files()["replaceme"]
    if f.exists(): em[e.spec.name] = round(json.load(open(f))["exact_match"] * 100, 2)
tables = analysis.series_tables(cfg, {e.spec.name: e.blocks for e in exps}, profiles, em)

In [ ]:
# 12. Diagnostic (fresh kernel: cell 1, then this): per-token grad_contrib split into question / answer tokens.
#     Candidate i = block (i+1, i+9). Expect ||g_out|| of question tokens to collapse in late windows.
print(analysis.grad_contrib_diagnostic(cfg, rm, windows=[4, 5, 21, 22], n_texts=16)
      .to_string(index=False, float_format=lambda x: f"{x:.4g}"))